# Data

Validate the sample metadata and RNA-seq processing provenance.


In [1]:
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

FRACTIONS = ('Input', 'Monosome', 'Polysome')


In [2]:
# Locate the project directory.


def find_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / 'data').is_dir():
            return path
    raise FileNotFoundError('Could not find the project data directory.')


ROOT = find_root(Path.cwd().resolve())
DATA_DIR = ROOT / 'data'


In [3]:
# Validate the sample metadata.

metadata = pd.read_csv(DATA_DIR / 'metadata.tsv', sep = '\t', dtype = 'string')
required = {'sampleId', 'familyId', 'condition'}
missing = required.difference(metadata.columns)

if missing:
    raise ValueError(f'Missing metadata columns: {sorted(missing)}')
if not metadata['sampleId'].is_unique:
    raise ValueError('Sample identifiers must be unique.')
if metadata.loc[:, sorted(required)].isna().any().any():
    raise ValueError('Required metadata values cannot be missing.')
if metadata['condition'].value_counts().to_dict() != {
    'ASD': 3,
    'Unaffected': 3,
}:
    raise ValueError('Expected three ASD and three unaffected samples.')
if metadata['familyId'].nunique() != 1:
    raise ValueError('Expected one family identifier.')

display(metadata)


,sampleId,familyId,condition
0,07c06,1077,Unaffected
1,07c13,1077,Unaffected
2,07c16,1077,Unaffected
3,1851,1077,ASD
4,1861,1077,ASD
5,1916,1077,ASD


In [4]:
# Extract processing information from the MultiQC reports.


def parse_multiqc(path: Path) -> dict[str, str]:
    text = path.read_text(encoding = 'utf-8', errors = 'ignore')
    pipeline = re.search(r'nf-core/rnaseq/tag/([^"/]+)', text)
    title = re.search(r'<title>([^<]+): MultiQC Report</title>', text)
    title = title.group(1).strip() if title else ''
    reference = re.search(r'(GRCh\d+_gencode_v\d+)', title)

    return {
        'fraction': path.parent.name,
        'nfCoreRnaseqVersion': pipeline.group(1) if pipeline else '',
        'referenceLabel': reference.group(1) if reference else '',
    }


paths = [DATA_DIR / fraction / 'multiqc_report.html' for fraction in FRACTIONS]
missing = [path for path in paths if not path.is_file()]

if missing:
    raise FileNotFoundError(f'Missing MultiQC reports: {missing}')

provenance = pd.DataFrame(map(parse_multiqc, paths))


In [5]:
# Confirm that all fractions used the same pipeline and reference.

if provenance[['nfCoreRnaseqVersion', 'referenceLabel']].eq('').any().any():
    raise ValueError('Processing provenance could not be recovered.')
if provenance['nfCoreRnaseqVersion'].nunique() != 1:
    raise ValueError('The nf-core/rnaseq versions are inconsistent.')
if provenance['referenceLabel'].nunique() != 1:
    raise ValueError('The reference labels are inconsistent.')

display(provenance)


,fraction,nfCoreRnaseqVersion,referenceLabel
0,Input,3.26.0,GRCh38_gencode_v50
1,Monosome,3.26.0,GRCh38_gencode_v50
2,Polysome,3.26.0,GRCh38_gencode_v50
